# 04 - async_execution 加速

如果某些 task 之间没有依赖,可以并行跑。
CrewAI 用 `Task(async_execution=True)` 标记,让 manager 调度时并行启动。

本 notebook 跑 3 个独立 task(3 种语言的问候),对比:

- 全部 sequential
- 全部 `async_execution=True`

总耗时差异会很明显(几乎 3 倍)。

**注意**:本 notebook 会发 6 次 LLM 调用,合计约 30-60 秒。


In [ ]:
import time
from alphaquant.infrastructure.llm import get_llm
from crewai import Agent, Task, Crew, Process

llm = get_llm(temperature=0.1)

spanish_agent = Agent(
    role="西语翻译",
    goal="用西语问候",
    backstory="你用西语说你好。",
    tools=[],
    llm=llm,
)
french_agent = Agent(
    role="法语翻译",
    goal="用法语问候",
    backstory="你用法语说你好。",
    tools=[],
    llm=llm,
)
japanese_agent = Agent(
    role="日语翻译",
    goal="用日语问候",
    backstory="你用日语说你好。",
    tools=[],
    llm=llm,
)


In [ ]:
seq_tasks = [
    Task(description="用西语说'你好'", expected_output="一个西语问候", agent=spanish_agent),
    Task(description="用法语说'你好'", expected_output="一个法语问候", agent=french_agent),
    Task(description="用日语说'你好'", expected_output="一个日语问候", agent=japanese_agent),
]
seq_crew = Crew(
    agents=[spanish_agent, french_agent, japanese_agent],
    tasks=seq_tasks,
    process=Process.sequential,
    verbose=False,
)

t0 = time.time()
seq_crew.kickoff(inputs={})
seq_t = time.time() - t0
print(f"Sequential 总耗时: {seq_t:.2f}s")


In [ ]:
async_tasks = [
    Task(description="用西语说'你好'", expected_output="一个西语问候", agent=spanish_agent, async_execution=True),
    Task(description="用法语说'你好'", expected_output="一个法语问候", agent=french_agent, async_execution=True),
    Task(description="用日语说'你好'", expected_output="一个日语问候", agent=japanese_agent, async_execution=True),
]
async_crew = Crew(
    agents=[spanish_agent, french_agent, japanese_agent],
    tasks=async_tasks,
    process=Process.sequential,  # process 仍然是 sequential,只是 task 自己标了 async
    verbose=False,
)

t0 = time.time()
async_crew.kickoff(inputs={})
async_t = time.time() - t0
print(f"Async 总耗时: {async_t:.2f}s")
print(f"加速比: {seq_t / async_t:.2f}x")


## 跟生产代码的对应

`AnalysisCrew` 用 `_ASYNC_TASK_INDICES = {0, 1, 2, 3, 4, 5, 6}`,
意思是前 7 个 task(数据 0-3 + 分析 4-6)并行跑,只有 idx 7 (`report_writer`)
是串行的 —— 因为它需要消费前 3 个分析 task 的输出。

在 `crews/analysis_crew.py` 的 `_build_tasks()` 中:

```python
async_execution=idx in self._ASYNC_TASK_INDICES,
```

如果你想让 `report_writer` 也并行,会出现两个问题:

1. 它需要 `tasks[4]`、`tasks[5]`、`tasks[6]` 的输出作为 context,显式注入可以解决
2. 但如果数据 task 还没跑完,manager 调度时可能会和它们抢资源,反而变慢

`_ASYNC_TASK_INDICES` 的设计原则:

- 没有数据依赖的 task 标 async
- 报告写作者必须最后跑(用 `context=[tasks[4], tasks[5], tasks[6]]` 显式注入)
- 显式控制并行边界,比全部 async 更稳定
